# Benchmark Runner
Recursively runs all benchmarks found under the `benchmarks` directory.
Results are saved to disk after each run, enabling **resume** if the notebook is interrupted.

In [ ]:
import os
import subprocess
import re
import json
import pandas as pd
import numpy as np
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
from pathlib import Path
from datetime import datetime
from typing import List, Dict, Optional

EXE_PATH      = os.path.join('build-bottom-up-cuda', 'bottom_up_cuda', 'Release', 'REI.exe')
BENCHMARKS_DIR = 'benchmarks/extended'
DEFAULT_COSTS  = [1, 1, 1, 1, 1]
MAX_COST       = 500
MAX_TIME       = 60 * 3
RESULTS_FILE   = 'bottom_up_results.json'

# ---------------------------------------------------------------------------
# Parsing
# ---------------------------------------------------------------------------

def parse_uint_list(raw: str) -> list:
    """Parse '[7653497]' or '[4 8 15]' into a list of ints."""
    return [int(x) for x in re.findall(r'\d+', raw)]


def parse_decomposition(block_text: str) -> list:
    """
    Parse the tree nodes that appear after the RE: line.
    Each node looks like:
        N - OP:
            root: [<uints>]
            left:  [<uints>]
            right: [<uints>]   <- optional
    Returns a list of dicts, one per node, in order.
    """
    nodes = []
    # Find every node header: '<index> - <OP>:'
    node_pattern = re.compile(
        r'^(\d+)\s*-\s*([A-Z]):\s*\n'
        r'(?:[^\n]*\n)*?'   # consumed lazily by the field patterns below
        , re.MULTILINE
    )
    # Split the block on node headers so each chunk belongs to one node
    header_re = re.compile(r'^(\d+)\s*-\s*([A-Z]):', re.MULTILINE)
    headers   = list(header_re.finditer(block_text))

    for i, hdr in enumerate(headers):
        # The chunk for this node runs from after its header line
        # to the start of the next header (or end of block)
        chunk_start = hdr.end()
        chunk_end   = headers[i + 1].start() if i + 1 < len(headers) else len(block_text)
        chunk       = block_text[chunk_start:chunk_end]

        node = {
            'index': int(hdr.group(1)),
            'op':    hdr.group(2),
        }

        m = re.search(r'root:\s*(\[[^\]]*\])', chunk)
        if m: node['root'] = parse_uint_list(m.group(1))

        m = re.search(r'left:\s*(\[[^\]]*\])', chunk)
        if m: node['left'] = parse_uint_list(m.group(1))

        m = re.search(r'right:\s*(\[[^\]]*\])', chunk)
        if m: node['right'] = parse_uint_list(m.group(1))

        nodes.append(node)

    return nodes


def parse_solution_block(block_text: str) -> dict:
    """Parse a single '--- N of M ---' block into a dict."""
    sol = {}

    m = re.search(r'All REs: (\d+)', block_text)
    if not m:
        m = re.search(r'^REs: (\d+)', block_text, re.MULTILINE)
    if m: sol['all_res'] = int(m.group(1))

    m = re.search(r'Unique REs: (\d+)', block_text)
    if m: sol['unique_res'] = int(m.group(1))

    m = re.search(r'Running Time: ([\d.]+) s', block_text)
    if m: sol['running_time_s'] = float(m.group(1))

    m = re.search(r'Cost: (\d+)', block_text)
    if m: sol['cost'] = int(m.group(1))

    m = re.search(r'^RE: "(.*)"', block_text, re.MULTILINE)
    if m: sol['re'] = m.group(1)

    sol['decomposition'] = parse_decomposition(block_text)

    return sol


def parse_output(output_text: str, benchmark_file: str) -> dict:
    """
    Parse everything that appears after '==== Output ===' in output_text.
    Returns a JSON-serialisable dict that includes the benchmark filename.
    """
    result = {
        'benchmark_file': benchmark_file,
        'success': False,
        'solutions': []
    }

    # Isolate the output section
    output_marker = '==== Output ==='
    idx = output_text.find(output_marker)
    if idx == -1:
        result['raw_output'] = output_text.strip()
        return result

    output_section = output_text[idx + len(output_marker):].strip()

    # Check for failure
    if re.search(r'failed to find any solution', output_section, re.IGNORECASE):
        result['raw_output'] = output_section
        return result

    # Count solutions
    found_match = re.search(r'Found (\d+) solution', output_section)
    if found_match:
        result['solutions_found'] = int(found_match.group(1))

    # Split into per-solution blocks on '----- N of M -----'
    blocks = re.split(r'-{5}\s*\d+ of \d+\s*-{5}', output_section)
    # blocks[0] is the header line ("Found N solution"), skip it
    solution_blocks = [b.strip() for b in blocks[1:] if b.strip()]

    for block in solution_blocks:
        sol = parse_solution_block(block)
        if sol:
            result['solutions'].append(sol)

    result['success'] = len(result['solutions']) > 0
    return result


# ---------------------------------------------------------------------------
# File discovery & sorting
# ---------------------------------------------------------------------------

def get_sort_val(file_path: str) -> tuple:
    import re
    parts = file_path.replace('\\', '/').split('/')
    sort_key = []
    for part in parts:
        sub_parts = []
        for chunk in re.split(r'(\d+)', part):
            if chunk:
                if chunk.isdigit():
                    sub_parts.append((0, int(chunk)))
                else:
                    sub_parts.append((1, chunk))
        sort_key.append(tuple(sub_parts))
    return tuple(sort_key)


def discover_benchmarks() -> List[str]:
    """Return sorted list of all .txt benchmark paths (relative to cwd)."""
    found = []
    for root, _, files in os.walk(BENCHMARKS_DIR):
        for f in files:
            if f.endswith('.txt'):
                rel = os.path.relpath(os.path.join(root, f)).replace('\\', '/')
                found.append(rel)
    return sorted(found, key=get_sort_val)


# ---------------------------------------------------------------------------
# Running
# ---------------------------------------------------------------------------

def run_benchmark(file_path: str, costs: list, max_cost: int) -> dict:
    cmd = [EXE_PATH, file_path] + [str(max_cost)] + [str(c) for c in costs]
    print(cmd)
    try:
        proc = subprocess.Popen(cmd, stdout=subprocess.PIPE,
                                stderr=subprocess.PIPE, text=True)
        try:
            stdout, stderr = proc.communicate(timeout=MAX_TIME)
        except subprocess.TimeoutExpired:
            proc.kill()
            stdout, stderr = proc.communicate()
            return {
                'benchmark_file': file_path,
                'success': False,
                'error': f'Timeout after {MAX_TIME}s',
                'solutions': []
            }

        if proc.returncode == 0:
            return parse_output(stdout, file_path)
        else:
            return {
                'benchmark_file': file_path,
                'success': False,
                'error': stderr.strip(),
                'solutions': []
            }
    except Exception as exc:
        return {
            'benchmark_file': file_path,
            'success': False,
            'error': str(exc),
            'solutions': []
        }


# ---------------------------------------------------------------------------
# Persistence
# ---------------------------------------------------------------------------

def load_results() -> Dict[str, dict]:
    """Load previously saved results, keyed by benchmark_file path."""
    if os.path.exists(RESULTS_FILE):
        with open(RESULTS_FILE, 'r') as fh:
            data = json.load(fh)
        # Support both list and dict formats
        if isinstance(data, list):
            return {r['benchmark_file']: r for r in data}
        return data
    return {}


def save_results(results: Dict[str, dict]):
    """Persist results dict to disk atomically via a temp file.

    Writes to a sibling .tmp file first, then replaces the real file in one
    os.replace() call.  This way a mid-write crash or notebook-close never
    leaves a corrupt results file on disk.
    """
    tmp_file = RESULTS_FILE + '.tmp'
    with open(tmp_file, 'w') as fh:
        json.dump(results, fh, indent=2)
    os.replace(tmp_file, RESULTS_FILE)


# ---------------------------------------------------------------------------
# Statistics helpers
# ---------------------------------------------------------------------------

def build_summary_df(results: Dict[str, dict]) -> pd.DataFrame:
    rows = []
    for r in results.values():
        if r['solutions']:
            best = min(r['solutions'], key=lambda s: s.get('cost', 9999))
            rows.append({
                'File':       r['benchmark_file'],
                'Status':     'success',
                'RE':         best.get('re', ''),
                'Cost':       best.get('cost'),
                '# Solutions': len(r['solutions']),
                'All REs':    best.get('all_res'),
                'Unique REs': best.get('unique_res'),
                'Time (s)':   best.get('running_time_s'),
            })
        else:
            rows.append({
                'File':       r['benchmark_file'],
                'Status':     'failed' if 'error' not in r else 'error',
                'RE':         r.get('error', 'not_found'),
                'Cost':       None,
                '# Solutions': 0,
                'All REs':    None,
                'Unique REs': None,
                'Time (s)':   None,
            })
    df = pd.DataFrame(rows)
    if not df.empty:
        df['_sort'] = df['File'].apply(get_sort_val)
        df = df.sort_values('_sort').drop(columns=['_sort']).reset_index(drop=True)
    return df


def print_statistics(df: pd.DataFrame):
    total      = len(df)
    successful = (df['Status'] == 'success').sum()
    failed     = (df['Status'] == 'failed').sum()
    errors     = (df['Status'] == 'error').sum()

    summary = pd.DataFrame({
        'Metric': ['Total Runs', 'Successful', 'No Solution Found', 'Errors'],
        'Value':  [total, successful, failed, errors]
    })
    print('### Summary')
    display(summary)

    succ_df = df[df['Status'] == 'success'][['Cost', '# Solutions', 'All REs', 'Unique REs', 'Time (s)']]
    if not succ_df.empty:
        print('\n### Detailed Statistics (Successful Runs)')
        display(succ_df.describe(percentiles=[.25, .5, .75]))


# ---------------------------------------------------------------------------
# UI
# ---------------------------------------------------------------------------

all_benchmarks = discover_benchmarks()
saved_results  = load_results()
remaining      = [b for b in all_benchmarks if b not in saved_results]

status_label = widgets.HTML(
    value=f'<b>Discovered:</b> {len(all_benchmarks)} benchmarks &nbsp;|&nbsp; '
          f'<b>Already done:</b> {len(saved_results)} &nbsp;|&nbsp; '
          f'<b>Remaining:</b> {len(remaining)}'
)

resume_checkbox = widgets.Checkbox(
    value=True,
    description='Resume (skip already-completed benchmarks)',
    indent=False,
    layout=widgets.Layout(width='400px')
)

reset_button = widgets.Button(
    description='Clear Saved Results',
    button_style='danger',
    tooltip='Delete benchmark_results.json and start fresh',
    icon='trash'
)

run_button = widgets.Button(
    description='Run All Benchmarks',
    button_style='success',
    tooltip='Run every .txt file found under benchmarks/',
    icon='play'
)

show_button = widgets.Button(
    description='Show Current Results',
    button_style='info',
    tooltip='Display results saved so far without running anything',
    icon='table'
)



progress_bar  = widgets.IntProgress(value=0, min=0, max=1,
                                     description='Progress:',
                                     layout=widgets.Layout(width='500px'))
progress_label = widgets.Label(value='')
output_area    = widgets.Output()


def refresh_status():
    saved = load_results()
    rem   = [b for b in all_benchmarks if b not in saved]
    status_label.value = (
        f'<b>Discovered:</b> {len(all_benchmarks)} benchmarks &nbsp;|&nbsp; '
        f'<b>Already done:</b> {len(saved)} &nbsp;|&nbsp; '
        f'<b>Remaining:</b> {len(rem)}'
    )
    return saved, rem


def on_reset_clicked(b):
    if os.path.exists(RESULTS_FILE):
        os.remove(RESULTS_FILE)
    with output_area:
        clear_output()
        print(f'Cleared {RESULTS_FILE}.')
    refresh_status()


def on_show_clicked(b):
    with output_area:
        clear_output()
        saved, _ = refresh_status()
        if not saved:
            print('No results saved yet.')
            return
        df = build_summary_df(saved)
        table_html = (
            "<div style='max-height:500px;overflow-y:auto;border:1px solid #ccc;'>"
            + df.to_html(index=False)
            + "</div>"
        )
        display(HTML(table_html))
        print()
        print_statistics(df)




def on_run_clicked(b):
    run_button.disabled   = True
    reset_button.disabled = True

    with output_area:
        clear_output()

        saved, _ = refresh_status()
        todo = [bm for bm in all_benchmarks if bm not in saved] \
               if resume_checkbox.value else list(all_benchmarks)

        if not todo:
            print('Nothing left to run. Uncheck "Resume" or clear saved results to rerun.')
            run_button.disabled  = False
            reset_button.disabled = False
            return

        total = len(todo)
        progress_bar.max   = total
        progress_bar.value = 0
        progress_label.value = f'0 / {total}'

        print(f'Running {total} benchmark(s){" (resuming)" if resume_checkbox.value else ""}…')
        start_time = datetime.now()

        for i, bm_path in enumerate(todo):
            progress_label.value = f'{i} / {total}  —  {bm_path}'
            result = run_benchmark(bm_path, DEFAULT_COSTS, MAX_COST)
            saved[bm_path] = result
            save_results(saved)          # persist after every run

            progress_bar.value = i + 1
            status_str = '✓' if result['success'] else '✗'
            progress_label.value = f'{i+1} / {total}  {status_str}  {bm_path}'

        elapsed = (datetime.now() - start_time).total_seconds()
        clear_output()
        print(f'Done! {total} benchmarks completed in {elapsed:.1f}s.')
        print(f'Results saved to {RESULTS_FILE}')

        df = build_summary_df(saved)
        table_html = (
            "<div style='max-height:500px;overflow-y:auto;border:1px solid #ccc;'>"
            + df.to_html(index=False)
            + "</div>"
        )
        display(HTML(table_html))
        print()
        print_statistics(df)

        refresh_status()

    run_button.disabled   = False
    reset_button.disabled = False


reset_button.on_click(on_reset_clicked)
show_button.on_click(on_show_clicked)
run_button.on_click(on_run_clicked)

display(widgets.VBox([
    status_label,
    widgets.HBox([resume_checkbox]),
    widgets.HBox([run_button, show_button, reset_button]),
    widgets.HBox([progress_bar, progress_label]),
    output_area
]))
